# MultiDevice Data Usage

This notebook explains how to use this repo's multi-device ETL pipeline, and how to navigate the data it produces.

Once loaded, data is organized as a nested structure: each device type (`device_type`) contains one or more devices (`device_id`), each of which holds one or more DataFrames (`df`) — and each holds its own variables (columns).

This notebook covers, in order:

1. **Loading the data** — running the ETL pipeline to populate the nested structure.
2. **Seeing what's available** — using `skim()` to get a quick overview of device types, devices, and dataframes.
3. **Retrieving device data** — using `get()` to pull out one or more types/devices/dataframes/variables.
5. **Preparing usable data** — using `merge()` to join tables within or across devices into a usable DataFrame — ready for analysis!

## Load All Data

The function `load()` triggers the full ETL (Extract, Transform, Load) pipeline, scanning every available device type and device ID and loading them into the nested data structure described above.

It's a one-liner — call it once at the start of a session to load everything

**Note**: This pipeline is currently extracts from a mounted Proton Drive folder, it can be upgraded to API calls per `device_id` instead.

In [ ]:
# import sys
# sys.path.insert(0, "..") # run at repo root
from utils import scroll_output # just for output presenation
from load import load # calls etl pipeline

MOUNT_PATH = "/home/yul/mnt/proton-data"

# data = load(MOUNT_PATH) # normal usage

data = scroll_output(lambda: load(MOUNT_PATH), plot_height="150px")

In [2]:
def show_device_keys():
    # See which device types are available 
    list_device_types = data.keys() 
    print(list_device_types)
    
    print("")
    # See available device IDs, per type
    list_device_ids = data["Atmotube"].keys()
    print(list_device_ids)

scroll_output(show_device_keys)

In [ ]:
# Alternatively, this is how to extract the raw, unprocessed data
# from src.etl.extract import extract_raw_data

# raw = extract_raw_data("/home/yul/mnt/proton-data")
# raw["Atmotube"]["C3CBE16AE294_01-May-2026_12-Jun-2026"]  # example

## See Available Devices

For each device type (`device_type`), the actual devices are indexed according to its device ID (`device_id`).

The **function `skim()`** quickly shows the available dataframes (`df`) per device — useful as a first orientation step before retrieving or merging anything.

It accepts: one or a list of each of: `device_type`, `device_id`, `df`, and column names — letting you scope the summary as broadly or narrowly as you like.
It returns: per available device, the shape (column count and row count) and range (start datetime to end datetime) of each dataframe — printed directly, not returned as a value.

In [ ]:
from utils import skim

def show_loaded_data_overview():
    # a. View ALL of the loaded data, devices, and device types
    # print("=== ALL ===")
    # skim(data)

    # b. View loaded data by device type
    print("\n=== PER TYPE ===")
    skim(data, ["Atmotube", "Fitbit"])

    print("")
    # c. View loaded data by device ID
    print("\n=== PER DEVICE ===")
    skim(
        data, "Atmotube",
        ["C7A595F09965_01-May-2026_12-Jun-2026", # how to select for devices
        "FD20113AFE41_01-May-2026_12-Jun-2026"]
    )

scroll_output(show_loaded_data_overview)

# (equivalent manual version, for comparison:)
# display_loaded_data({"Atmotube": {"C3CBE16AE294_01-May-2026_12-Jun-2026": data["Atmotube"]["C3CBE16AE294_01-May-2026_12-Jun-2026"]}})

When `skim()` used to inspect one or more `df`, it returns its available variables' (AKA columns) row length and data type (AKA dtype).

In [ ]:
from utils import skim

def show_dataframe_and_variable_views():
    # d. View loaded data by dataframe
    print("=== PER DATAFRAME ===")
    # skim(data, "Atmotube", "C7A595F09965_01-May-2026_12-Jun-2026", "pm")
    skim(data, "Atmotube", "C7A595F09965_01-May-2026_12-Jun-2026", ["pm", "weather"])
    
    print("")
    # e. View loaded data by variable
    print("=== PER VARIABLE ===")
    # skim(data, "Atmotube", "C7A595F09965_01-May-2026_12-Jun-2026", "pm", "pm10_ugm3_atm")
    skim(data, "Atmotube", "C7A595F09965_01-May-2026_12-Jun-2026", "pm", ["pm2_5_ugm3_atm", "pm10_ugm3_atm"])

scroll_output(show_dataframe_and_variable_views)

### Structural Relationship of the `Data` 

Below represents the hierarchical relationship between the data based on device type, devices, and each device's dataframes and their variables. 

They are all retrievable as a dictionary (`dict`) of each other. 

In [ ]:
import json
from utils import get

def show_device_structure():
    print("Available Device Types:")
    print(json.dumps({k: type(v).__name__ for k, v in data.items()}, indent=2))
    
    print("")
    print("\nDevices in Atmotube:")
    print(json.dumps({k: type(v).__name__ for k, v in data["Atmotube"].items()}, indent=2))

    print("")
    print("Dataframes in a Device:")
    device = get(data, "Atmotube", "C7A595F09965_01-May-2026_12-Jun-2026")
    if device is None:
        print("⚠️ Device not found at that path")
    else:
        print(json.dumps({k: type(v).__name__ for k, v in device.items()}, indent=2))

scroll_output(show_device_structure)


### Un/Wrapped Nature of the `Data`

Due to the hierarchical structure of the data; the raw content of each `device_id` is wrapped inside a `data` key — storing the usable Dataframes as dicts. 

*Note*: `get()` already unwraps and returns the actual Dataframe.

In [ ]:
import json
from utils import get

def show_data_structure():
    def describe_one_level(d: dict) -> dict:
        out = {}
        for k, v in d.items():
            if isinstance(v, dict):
                out[k] = {inner_k: type(inner_v).__name__ for inner_k, inner_v in v.items()}
            else:
                out[k] = type(v).__name__
        return out

    print("Wrapped Contents of 1 Device (raw data is wrapped inside 'data' key):")
    raw_content = data["Atmotube"]["C7A595F09965_01-May-2026_12-Jun-2026"]
    print(json.dumps(describe_one_level(raw_content), indent=2))
    print("")

    print("Unwrapped Contents of 1 Device (removed 'data' key):")
    device = get(data, "Atmotube", "C7A595F09965_01-May-2026_12-Jun-2026") # get() already unwraps this internally
    if device is None:
        print("⚠️ Device not found at that path")
    else:
        print(json.dumps({k: type(v).__name__ for k, v in device.items()}, indent=2))

scroll_output(show_data_structure)

## Retrieving Data per Devices

Since any device and its contents are kept as nested dictionaries (`dict`), navigating to retrieve data at any level (`device_type`, `device_id`, `df`, and/or column) is possible.

The **function `get()`** navigates that nested structure and retrieves one or more device types, device IDs, their dataframes, and included variables (columns) — this is the function you'll typically reach for once `skim()` has shown you what's available.

It returns data in one of two shapes, depending on the specified level of the hierarchy in the given argument:

1. It accepts: a *list* for any level (selecting multiple types, devices, dataframes, or variables).
   It returns: a *nested* dictionary, structured down to that level — requiring unwrapping (e.g. with `unwrap()`) to access its contents.

2. It accepts: a *single* value (selecting one type, device, dataframe, and/or variable).
   It returns: a *flattened* dictionary, collapsed at that level — directly accessing its contents.

*Reminder*: regardless if its nested or flattened, it returns the unwrapped DataFrames.

### Per Devices/Types

In [ ]:
from utils import get

def show_loaded_data_by_get():
    # a. View ALL of the loaded data, devices, and device types
    # all_data = get(data) # Returns input unchanged (nested dict) 

    # b. View loaded data by 1/more device type
    print("=== 1/MORE TYPES ===")
    # device_types = get(data, "Atmotube")
    device_types = get(data, 
                    ["Fitbit", # nested dict per device_type
                    "Ponyopi"]) 
    print(device_types)

    print("")
    # c. View loaded data by device ID
    print("=== 1/MORE DEVICES, 1 TYPE ===")
    
    device_ids = get(data, "Atmotube",
                        ["C7A595F09965_01-May-2026_12-Jun-2026", # flattened dict per device_id 
                        "DB7A737B8CA0_01-May-2026_12-Jun-2026"])
    print(device_ids)

scroll_output(show_loaded_data_by_get)

# (equivalent manual version:)
# atmotube_device = data["Atmotube"]["C3CBE16AE294_01-May-2026_12-Jun-2026"]["data"]
# this can only do one device and type at a time

### Per Dataframes/Devices

In [ ]:
from utils import get

def show_loaded_data_by_dataframe():
    # d. View loaded data by 1/more dataframe

    print("")
    print("=== 1 DATAFRAMES, 1 DEVICE ===")
    ## d.1. One Df, One Device, One Type
    atmotube_df = get(data, "Atmotube", "C7A595F09965_01-May-2026_12-Jun-2026", "pm") # flattened per df
    print(atmotube_df)

    print("")
    print("=== 1/MORE DATAFRAMES, 1 DEVICE ===")
    ## d.2. Two Dfs, One Device, One Type
    atmotube_dfs = get(data, "Atmotube", "C7A595F09965_01-May-2026_12-Jun-2026", 
                    ["pm", # flattened per df
                    "weather"])
    print(atmotube_dfs)


    print("")
    print("=== 1/MORE DATAFRAMES, 1/MORE DEVICES ===")
    ## d.3. Two Dfs, Two Devices, One Type
    atmotube_devices_dfs = get(data, "Atmotube", 
                            ["C7A595F09965_01-May-2026_12-Jun-2026", # flattened per device_id
                            "DB7A737B8CA0_01-May-2026_12-Jun-2026"], 
                            ["pm", "weather"]) 
    print(atmotube_devices_dfs)

scroll_output(show_loaded_data_by_dataframe)

# (equivalent manual version:)
# atmotube_device = data["Atmotube"]["C3CBE16AE294_01-May-2026_12-Jun-2026"]["data"]["pm"]
# this can only do one df per device at a time

### Per Variables/Devices

In [ ]:
from utils import get

def show_loaded_data_by_variable():
    # e. View loaded data by 1/more variable

    ## e.1. One Variable, One Df, One Device, One Type
    # atmotube_col = get(data, "Atmotube", "C7A595F09965_01-May-2026_12-Jun-2026", "pm", "pm10_ugm3_atm")

    print("")
    print("=== 1/MORE VARIBLE, 1 DATAFRAME, 1 DEVICE ===")
    ## e.2. Two Variables, One Df, One Device, One Type
    atmotube_cols = get(data, "Atmotube", "C7A595F09965_01-May-2026_12-Jun-2026", "pm", 
                ["pm2_5_ugm3_atm", # flattened per col
                "pm10_ugm3_atm"])
    print(atmotube_cols)

    print("")
    print("=== 1/MORE VARIBLE, 1/MORE DATAFRAMES, 1 DEVICE ===")
    ## e.3. One Variable, One Df, Two Device, One Type
    atmotube_dfs_cols = get(data, "Atmotube",
                            ["C7A595F09965_01-May-2026_12-Jun-2026",
                            "DB7A737B8CA0_01-May-2026_12-Jun-2026"],
                            "pm", "pm2_5_ugm3_atm")
    print(atmotube_dfs_cols)

    print("")
    print("=== 1/MORE VARIBLE, 1/MORE DATAFRAMES, 1/MORE DEVICE ===")
    ## e.4. Two Variables, Two Dfs, Two Devices, One Type
    atmotube_devices_dfs_cols = get(data, "Atmotube",
                            ["C7A595F09965_01-May-2026_12-Jun-2026",
                            "DB7A737B8CA0_01-May-2026_12-Jun-2026"],
                            ["pm", "weather"],
                            ["pm2_5_ugm3_atm", "temp_c"]
    )
    print(atmotube_devices_dfs_cols)

scroll_output(show_loaded_data_by_variable)

# (equivalent manual version:)
# atmotube_device = data["Atmotube"]["C3CBE16AE294_01-May-2026_12-Jun-2026"]["data"]["pm"]["pm2_5_ugm3_atm"] 
# this can only do one variable per df per device at a time

## Preparing Data to Use

An actual workflow would look like: 
>`get()` to retrieve data from 1/more device → `merge()` to create 1/more usable data table/s!

Upon retrieval, a device's data and its selected dataframes/variables, are already unwrapped and ready to be merged into an table. 
It would be very useful to compare selected variables from different dataframes within *one device*, or across *multiple devices*. 

The **function `merge()`** takes one or more selected variables/dataframes/devices/types (retrieved and returned via `get()`) and joins them all on`"datetime"` into a single wide DataFrame.
 
It accepts either:
1.  *One device* — no column suffixing, since there's no ambiguity about which device a column came from.
2.  *Multiple devices* — every column (except `datetime`) gets suffixed with `_<device_name>`, so overlapping column names (e.g. both devices have `temp_c`) don't collide after the join.

It returns: one wide, merged DataFrame — ready for analysis.

Optional arguments:
- `df_names`: a list of table names (e.g. `["pm", "weather"]`) to limit the merge to specific tables, rather than merging every table available.
- `how`: the join type passed to `pandas.merge` — `"outer"` by default, keeping all rows from every table even where one device/table is missing data for a given timestamp. Other options: `"left"`, `"right"`, `"inner"`, `"cross"`, `"left_anti"`, `"right_anti"`.

**Note**: passing both a positional dict *and* named devices, or multiple unnamed positional dicts, will raise an error — the function needs a name for each device whenever there's more than one, so it knows what to suffix columns with.

### For One Device

In [ ]:
from utils import merge, get

def show_one_device_all():
    print("\n=== TOTAL DATA, 1 DEVICE ===")
    device_total = get(data, "Atmotube", "C7A595F09965_01-May-2026_12-Jun-2026")
    device_total = merge({"": device_total})
    # OR, in one line:
    # device_total = merge({"": get(data, "Atmotube", "C7A595F09965_01-May-2026_12-Jun-2026")})

    display(device_total.head())  

scroll_output(show_one_device_all)

,datetime,longitude,latitude,timezone,altitude,pm1_0_ugm3_atm,pm2_5_ugm3_atm,pm10_ugm3_atm,aqs_total,temp_c,...,tvoc_ppm,nox_index,co2_ppm,battery_phone_pct,motion_phone_bool,gps_phone_bool,charge_phone_bool,cooldown_phone_bool,position_error_m,user_notes
0,2026-05-30 14:57:00+00:00,NaN,NaN,<NA>,NaN,16.3,17.1,17.1,76,31.5,...,0.093,1.0,691.0,24,False,<NA>,False,False,NaN,NaN
1,2026-05-30 14:58:00+00:00,88.35042,22.50010,Asia/Kolkata,0.0,14.8,15.6,15.6,74,31.6,...,0.093,1.0,734.0,24,True,True,False,False,9.0,NaN
2,2026-05-30 14:59:00+00:00,88.35043,22.50008,Asia/Kolkata,0.0,15.6,16.4,16.4,74,31.5,...,0.095,1.0,727.0,24,True,True,False,False,7.0,NaN
3,2026-05-30 15:00:00+00:00,88.35045,22.50014,Asia/Kolkata,0.0,14.7,15.4,15.4,74,31.5,...,0.145,1.0,718.0,24,True,True,False,False,8.0,NaN
4,2026-05-30 15:01:00+00:00,88.35052,22.50025,Asia/Kolkata,0.0,17.5,18.4,18.4,75,31.4,...,0.181,1.0,706.0,24,True,True,False,False,6.0,NaN


In [ ]:
from utils import merge, get

def show_one_device_dfs():
    print("\n=== SELECT DATAFRAMES, 1 DEVICE ===")
    device_dfs = get(data, "Atmotube", "C7A595F09965_01-May-2026_12-Jun-2026")
    device_dfs = merge({"": device_dfs}, df_names=["pm", "weather"])
    # OR, in one line:
    # device_dfs = merge({"": get(data, "Atmotube", "C7A595F09965_01-May-2026_12-Jun-2026")}, df_names=["pm", "weather"])

    display(device_dfs.head())
    
scroll_output(show_one_device_dfs)

,datetime,pm1_0_ugm3_atm,pm2_5_ugm3_atm,pm10_ugm3_atm,aqs_total,temp_c,hum_pct,press_hpa
0,2026-05-30 14:57:00+00:00,16.3,17.1,17.1,76,31.5,66.0,1004.0
1,2026-05-30 14:58:00+00:00,14.8,15.6,15.6,74,31.6,67.0,1004.0
2,2026-05-30 14:59:00+00:00,15.6,16.4,16.4,74,31.5,66.0,1004.1
3,2026-05-30 15:00:00+00:00,14.7,15.4,15.4,74,31.5,66.0,1004.1
4,2026-05-30 15:01:00+00:00,17.5,18.4,18.4,75,31.4,67.0,1004.7


In [ ]:
from utils import merge, get

def show_one_device_cols():
    print("\n=== SELECT VARIABLES, 1 DEVICE ===")
    device_dfs = merge(
        {"": get(
            data, "Atmotube", "C7A595F09965_01-May-2026_12-Jun-2026",
            ["pm", "weather"],                        # df_key: which tables
            ["pm2_5_ugm3_atm", "temp_c"]                # col: which columns
        )}
    )
    display(device_dfs.head())

scroll_output(show_one_device_cols)

,datetime,pm2_5_ugm3_atm,temp_c
0,2026-05-30 14:57:00+00:00,17.1,31.5
1,2026-05-30 14:58:00+00:00,15.6,31.6
2,2026-05-30 14:59:00+00:00,16.4,31.5
3,2026-05-30 15:00:00+00:00,15.4,31.5
4,2026-05-30 15:01:00+00:00,18.4,31.4


### For Multiple Devices

In [ ]:
from utils import merge, get

def show_two_device_all():
    print("\n=== TOTAL DATA, 1/MORE DEVICES ===")
    d1 = get(data, "Atmotube", "C7A595F09965_01-May-2026_12-Jun-2026")
    d2 = get(data, "Atmotube", "DB7A737B8CA0_01-May-2026_12-Jun-2026")

    device_total = merge({"d1": d1, "d2": d2})
    display(device_total.head())

scroll_output(show_two_device_all)

,datetime,longitude_d1,latitude_d1,timezone_d1,altitude_d1,pm1_0_ugm3_atm_d1,pm2_5_ugm3_atm_d1,pm10_ugm3_atm_d1,aqs_total_d1,temp_c_d1,...,tvoc_ppm_d2,nox_index_d2,co2_ppm_d2,battery_phone_pct_d2,motion_phone_bool_d2,gps_phone_bool_d2,charge_phone_bool_d2,cooldown_phone_bool_d2,position_error_m_d2,user_notes_d2
0,2026-05-27 12:37:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,...,NaN,NaN,NaN,88.0,False,<NA>,True,False,NaN,NaN
1,2026-05-27 12:38:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,...,0.018,1.0,NaN,89.0,False,<NA>,True,False,NaN,NaN
2,2026-05-27 12:39:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,...,0.049,1.0,386.0,90.0,False,<NA>,True,False,NaN,NaN
3,2026-05-27 12:40:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,...,0.079,1.0,391.0,90.0,False,<NA>,True,False,NaN,NaN
4,2026-05-27 12:41:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,...,0.090,1.0,391.0,91.0,False,<NA>,True,False,NaN,NaN


In [ ]:
from utils import merge, get

def show_two_device_dfs():
    print("\n=== SELECT DATAFRAMES, 1/MORE DEVICES ===")
    # 1. Get the devices to analyze
    d1 = get(data, "Atmotube", "C7A595F09965_01-May-2026_12-Jun-2026")
    d2 = get(data, "Atmotube", "DB7A737B8CA0_01-May-2026_12-Jun-2026")

    # 2. Merge the devices and select the dataframes of interest
    device_dfs = merge(
        {"C7A595F09965": d1, "DB7A737B8CA0": d2},
        df_names=["pm", "weather"]
    )
    display(device_dfs.head())

scroll_output(show_two_device_dfs)

,datetime,pm1_0_ugm3_atm_d1,pm2_5_ugm3_atm_d1,pm10_ugm3_atm_d1,aqs_total_d1,temp_c_d1,hum_pct_d1,press_hpa_d1,pm1_0_ugm3_atm_d2,pm2_5_ugm3_atm_d2,pm10_ugm3_atm_d2,aqs_total_d2,temp_c_d2,hum_pct_d2,press_hpa_d2
0,2026-05-27 12:37:00+00:00,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,100,NaN,NaN,1005.4
1,2026-05-27 12:38:00+00:00,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,20.2,21.4,21.4,75,NaN,NaN,1005.4
2,2026-05-27 12:39:00+00:00,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,95,NaN,NaN,1005.4
3,2026-05-27 12:40:00+00:00,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,92,NaN,NaN,1005.4
4,2026-05-27 12:41:00+00:00,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,91,NaN,NaN,1005.3


In [ ]:
from utils import merge, get

def show_two_device_cols():
    print("\n=== SELECT VARIABLES, 1/MORE DEVICES ===")
    # 1. Get the devices to analyze
    d1 = get(data, "Atmotube", "C7A595F09965_01-May-2026_12-Jun-2026")
    d2 = get(data, "Atmotube", "DB7A737B8CA0_01-May-2026_12-Jun-2026")

    # 2. Merge the devices and select the columns of interest
    device_cols = merge(
        {"C7A595F09965": d1, "DB7A737B8CA0": d2},
        df_names=["pm", "weather"]
    )
    device_cols = device_cols[[
        "datetime",
        "pm2_5_ugm3_atm_C7A595F09965", "pm2_5_ugm3_atm_DB7A737B8CA0",
        "temp_c_C7A595F09965", "temp_c_DB7A737B8CA0"
    ]]

    # 3. OPTIONAL: Drop missing values
    # device_cols = device_cols.dropna(subset=["pm2_5_ugm3_atm_C7A595F09965", "pm2_5_ugm3_atm_DB7A737B8CA0"]) # drops a row only if the subsetted columns have NaN
    # OR
    # device_cols = device_cols.dropna(how="all") #  drops a row only if all columns have NaN
                                    # , subset=["pm2_5_ugm3_atm_C7A595F09965", "pm2_5_ugm3_atm_DB7A737B8CA0", "temp_c_C7A595F09965", "temp_c_DB7A737B8CA0"]) # but all looks at the specified subset, if included
    device_cols = device_cols.dropna()  # drops a row if any column has NaN
    
    display(device_cols.head())

scroll_output(show_two_device_cols)

,datetime,pm2_5_ugm3_atm_d1,pm2_5_ugm3_atm_d2,temp_c_d1,temp_c_d2
4460,2026-05-30 14:57:00+00:00,17.1,16.3,31.5,30.7
4461,2026-05-30 14:58:00+00:00,15.6,16.5,31.6,30.7
4462,2026-05-30 14:59:00+00:00,16.4,16.5,31.5,30.7
4463,2026-05-30 15:00:00+00:00,15.4,16.9,31.5,30.8
4464,2026-05-30 15:01:00+00:00,18.4,16.7,31.4,30.8
